# CWA vs Baselines: Depth Sweep + Fixed-J Ablation on CIFAR-10

This notebook demonstrates the **Curie-Weiss Activation (CWA)** — a novel within-sample mean-field activation function with a learnable coupling parameter J — compared against five baselines (ReLU, GELU, SELU, CompetingNonlinearities, GELU+LN) on CIFAR-10 image classification.

**Experiment A — Depth Sweep:** MLPs at depths {6, 10, 20} × 6 activations × 3 seeds, 25 epochs each on GPU.  
**Experiment B — Fixed-J Ablation:** J frozen at {0.1, 0.3, 0.5, 0.7, 0.9} plus learned J at depth=10, testing the gradient-stability mechanism independently.

**Verdict: PARTIAL_CONFIRM.** The gradient stability mechanism is confirmed via fixed-J ablation (J=0.7 achieves grad_ratio=0.364, J=0.9 achieves 0.410 — both below 2.0 threshold), but the accuracy advantage claimed by the hypothesis is not observed.

This demo:
1. Loads precomputed results from the full experiment
2. Visualizes the key findings (gradient ratios, accuracy by depth, fixed-J ablation)
3. Runs a minimal live training demo (CWA vs GELU, 2 epochs) to show the code works

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT pre-installed on Colab
_pip('loguru==0.7.2')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0')
    # torch 2.9.0 pairs with torchvision 0.24.0 (minor offset of 15)
    _pip('torch==2.9.0+cpu', 'torchvision==0.24.0+cpu',
         '--index-url', 'https://download.pytorch.org/whl/cpu')

## Imports

Standard imports from the original script, plus matplotlib for visualization.

In [ ]:
import sys
import os
import json
import math
import gc
from pathlib import Path

from loguru import logger
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

import resource
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import numpy as np
from scipy import stats
from torch.optim.lr_scheduler import CosineAnnealingLR

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

print(f"PyTorch: {torch.__version__}, NumPy: {np.__version__}")

## Data Loading

Load precomputed experiment results from GitHub (with local fallback).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-2f6f35-curie-weiss-activation-formal-verificati/main/round-2/experiment-1/demo/mini_demo_data.json"
import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded: {data['metadata']['experiment_name']}")
print(f"Verdict: {data['metadata']['verdict']}")
print(f"Reason: {data['metadata']['verdict_reason']}")
print(f"Examples: {len(data['datasets'][0]['examples'])}")

## Configuration

All tunable parameters for the **live demo run** are here. The minimum values produce results quickly; the original full-experiment values are shown in comments.

In [ ]:
# ── Hardware ────────────────────────────────────────────────────────────────────
HAS_GPU = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAS_GPU else "cpu")
print(f"Device: {DEVICE}")

# ── Demo training config (MINIMUM values for quick demo run) ────────────────────
DEMO_EPOCHS    = 2      # original: 25
DEMO_HIDDEN    = 64     # original: 256
DEMO_BATCH     = 256    # original: 256
DEMO_DEPTH     = 6      # original: 6, 10, 20
DEMO_LR        = 1e-3   # original: 1e-3
DEMO_SEED      = 0      # original: 0, 1, 2
DEMO_K_MAX     = 10     # original: 50  (CWA fixed-point iterations)

# Activations to compare in demo
DEMO_ACTIVATIONS = ["cwa", "gelu"]  # original also: relu, selu, competing_nl, gelu_ln

# ── Full experiment config (reference only, not run in demo) ────────────────────
# EXP_A_ACTIVATIONS = ["cwa", "relu", "gelu", "selu", "competing_nl", "gelu_ln"]
# EXP_A_DEPTHS = [6, 10, 20]
# EXP_A_SEEDS  = [0, 1, 2]
# EXP_B_FIXED_J = [0.1, 0.3, 0.5, 0.7, 0.9]
# EPOCHS = 25

## Hardware Detection

Detect available CPUs and RAM from cgroup limits (container-aware).

In [ ]:
# ─── Hardware detection ────────────────────────────────────────────────────────
def _detect_cpus() -> int:
    try:
        parts = Path("/sys/fs/cgroup/cpu.max").read_text().split()
        if parts[0] != "max":
            return math.ceil(int(parts[0]) / int(parts[1]))
    except (FileNotFoundError, ValueError):
        pass
    try:
        return len(os.sched_getaffinity(0))
    except (AttributeError, OSError):
        pass
    return os.cpu_count() or 1

def _container_ram_gb() -> float:
    for p in ["/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"]:
        try:
            v = Path(p).read_text().strip()
            if v != "max" and int(v) < 1_000_000_000_000:
                return int(v) / 1e9
        except (FileNotFoundError, ValueError):
            pass
    return 32.0

NUM_CPUS = _detect_cpus()
TOTAL_RAM_GB = _container_ram_gb()

logger.info(f"Hardware: {NUM_CPUS} CPUs, GPU={HAS_GPU}, RAM={TOTAL_RAM_GB:.1f}GB")
if HAS_GPU:
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    logger.info(f"GPU: {torch.cuda.get_device_properties(0).name}, VRAM={VRAM_GB:.1f}GB")

## CWA Layer Definition

The Curie-Weiss Activation computes:
$$y_i = \tanh(x_i + J \cdot m^*), \quad m^* = \frac{1}{n}\sum_j \tanh(x_j + J \cdot m^*)$$

Where $m^*$ is the mean-field fixed point found iteratively (K_max steps). The coupling J controls whether the system is in the **sub-critical** ($J \cdot \bar{s} < 1$) or **near-critical** ($J \cdot \bar{s} \geq 0.8$) regime.

**Key insight:** Near-critical J values (0.7–0.9) produce more balanced gradient flow across layers than standard activations.

In [ ]:
# ─── CWA Layer ─────────────────────────────────────────────────────────────────
class CWALayer(nn.Module):
    """Curie-Weiss Activation: y_i = tanh(x_i + J * m*), where m* = mean(tanh(x + J*m*))."""

    def __init__(self, fixed_J=None, K_max=50):
        super().__init__()
        self.K_max = K_max
        self.fixed_J = fixed_J
        if fixed_J is None:
            # Learnable: J = sigmoid(J_raw), init J_raw=0 => J=0.5
            self.J_raw = nn.Parameter(torch.zeros(1))
        else:
            self.register_buffer("J_buf", torch.tensor([float(fixed_J)], dtype=torch.float32))
        self._last_J_s_bar = 0.0
        self._last_K = 0
        self._last_mode = "unrolled"

    def get_J(self) -> torch.Tensor:
        if self.fixed_J is None:
            return torch.sigmoid(self.J_raw)
        else:
            return self.J_buf

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        J = self.get_J()
        J_val = J.item()
        delta = 1e-4 * (1.0 - J_val)  # tolerance from Lean Theorem 3

        # Phase 1: converge m* without grad (cheap, no graph)
        with torch.no_grad():
            m = torch.zeros(x.shape[:-1] + (1,), device=x.device, dtype=x.dtype)
            K_conv = self.K_max
            for k in range(self.K_max):
                m_new = torch.mean(torch.tanh(x + J_val * m), dim=-1, keepdim=True)
                if (m_new - m).abs().max().item() < delta:
                    m = m_new
                    K_conv = k + 1
                    break
                m = m_new
            m_star = m  # shape: (batch, 1)

        # Phase 2: compute s_bar for mode selection
        with torch.no_grad():
            z_star = x + J_val * m_star
            s_bar = (1.0 - torch.tanh(z_star) ** 2).mean().item()
        J_s_bar = J_val * s_bar
        self._last_J_s_bar = J_s_bar
        self._last_K = K_conv

        if J_s_bar >= 0.8:
            # IFT BRANCH: detach m_star, gradient flows through x directly (s_k term)
            # and through J via detached computation
            self._last_mode = "ift"
            m_star_detached = m_star.detach()
            # For x gradient: dy_i/dx_k = sech^2(z_i) * delta_ik (direct term only)
            # IFT chain adds s_i * J/(n*(1-J*s_bar)) * sum_k s_k -- small correction
            y = torch.tanh(x + J_val * m_star_detached)
        else:
            # UNROLLED BRANCH: 3 tracked steps from detached m_star
            self._last_mode = "unrolled"
            m = m_star.detach()
            steps = min(K_conv, 3)
            for _ in range(steps):
                m = torch.mean(torch.tanh(x + J * m), dim=-1, keepdim=True)
            y = torch.tanh(x + J * m)

        return y

## Baseline Activations

**CompetingNonlinearities:** A quenched random mixture where each neuron is randomly assigned (at initialization, then frozen) to either Swish (p=0.83) or Tanh.

In [ ]:
# ─── Baselines ──────────────────────────────────────────────────────────────────
class CompetingNonlinearities(nn.Module):
    """Quenched random mixture: each neuron fixed as Swish (p=0.83) or Tanh."""

    def __init__(self, n_neurons: int, p_c: float = 0.83):
        super().__init__()
        mask = (torch.rand(1, n_neurons) < p_c).float()
        self.register_buffer("mask", mask)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        swish_out = x * torch.sigmoid(x)
        tanh_out = torch.tanh(x)
        return self.mask * swish_out + (1.0 - self.mask) * tanh_out

## MLP Factory

Builds an MLP with `depth` activation layers:
```
Linear(n_in, hidden) -> [LN] -> Act -> Linear(hidden, hidden) -> [LN] -> Act -> ... -> Linear(hidden, n_out)
```
Total: depth+1 linear layers, depth activation layers.

In [ ]:
# ─── MLP Factory ───────────────────────────────────────────────────────────────
def make_activation(activation: str, hidden: int, fixed_J=None, p_c: float = 0.83) -> nn.Module:
    if activation == "cwa":
        return CWALayer(fixed_J=fixed_J, K_max=DEMO_K_MAX)
    elif activation == "relu":
        return nn.ReLU()
    elif activation == "gelu":
        return nn.GELU()
    elif activation == "selu":
        return nn.SELU()
    elif activation == "competing_nl":
        return CompetingNonlinearities(hidden, p_c=p_c)
    elif activation == "gelu_ln":
        return nn.GELU()  # LN inserted separately in build_mlp
    else:
        raise ValueError(f"Unknown activation: {activation}")


def build_mlp(
    depth: int,
    hidden: int = 256,
    n_in: int = 3072,
    n_out: int = 10,
    activation: str = "cwa",
    fixed_J=None,
    use_ln: bool = False,
    p_c: float = 0.83,
) -> nn.Sequential:
    layers = [nn.Linear(n_in, hidden)]
    for i in range(depth):
        if use_ln:
            layers.append(nn.LayerNorm(hidden))
        act = make_activation(activation, hidden, fixed_J, p_c)
        layers.append(act)
        if i < depth - 1:
            layers.append(nn.Linear(hidden, hidden))
    layers.append(nn.Linear(hidden, n_out))

    model = nn.Sequential(*layers)

    # SELU requires LeCun initialization
    if activation == "selu":
        for m in model.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=1.0 / math.sqrt(m.in_features))
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    return model

## Gradient Ratio Measurement

Measures the ratio of gradient norms at the first vs last linear layer:
$$\text{grad\_ratio} = \frac{|\log \|\nabla W_1\||}{|\log \|\nabla W_L\||}$$

A ratio near 1.0 indicates balanced gradient flow. Vanishing gradients → ratio < 1; exploding → ratio > 1.

In [ ]:
# ─── Gradient ratio ─────────────────────────────────────────────────────────────
def measure_gradient_ratios(model, loader, loss_fn, device):
    """Measure gradient norms for W_1 (first linear) and W_L (last linear)."""
    model.zero_grad()
    x, y = next(iter(loader))
    x, y = x.to(device), y.to(device)
    out = model(x)
    loss = loss_fn(out, y)
    loss.backward()

    linear_layers = [m for m in model.modules() if isinstance(m, nn.Linear)]
    W_first = linear_layers[0]
    W_last = linear_layers[-1]

    eps = 1e-12
    gf = W_first.weight.grad.norm().item() if W_first.weight.grad is not None else eps
    gl = W_last.weight.grad.norm().item() if W_last.weight.grad is not None else eps

    ratio = abs(math.log(gf + eps)) / abs(math.log(gl + eps))
    model.zero_grad()
    return ratio, gf, gl

## CIFAR-10 Data Loaders

CIFAR-10: 50k train / 10k test, 32×32 RGB images, 10 classes. Flattened to 3072-dim vectors for the MLP.

In [ ]:
# ─── Training loop ──────────────────────────────────────────────────────────────
def get_cifar10_loaders(batch_size: int = 256):
    transform = T.Compose([
        T.ToTensor(),
        T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        T.Lambda(lambda x: x.view(-1)),
    ])
    cifar_dir = Path("./cifar_data")
    cifar_dir.mkdir(exist_ok=True)
    train_ds = torchvision.datasets.CIFAR10(str(cifar_dir), train=True, download=True, transform=transform)
    test_ds  = torchvision.datasets.CIFAR10(str(cifar_dir), train=False, download=True, transform=transform)
    num_workers = min(2, NUM_CPUS)
    train_loader = torch.utils.data.DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=HAS_GPU
    )
    test_loader = torch.utils.data.DataLoader(
        test_ds, batch_size=512, shuffle=False, num_workers=num_workers, pin_memory=HAS_GPU
    )
    return train_loader, test_loader

logger.info("Downloading CIFAR-10 (cached after first run)...")
train_loader, test_loader = get_cifar10_loaders(batch_size=DEMO_BATCH)
logger.info("CIFAR-10 ready")

## Training Function

Trains one MLP configuration. Uses Adam + CosineAnnealingLR. Records gradient ratios at epoch 5 and final epoch, and CWA-specific statistics (J·s̄ trajectory, K convergence steps).

In [ ]:
def train_one_config(
    depth: int,
    activation_name: str,
    seed: int,
    fixed_J=None,
    epochs: int = 25,
    hidden: int = 256,
    batch: int = 256,
    lr: float = 1e-3,
    device: torch.device = DEVICE,
) -> dict:
    torch.manual_seed(seed)
    np.random.seed(seed)

    train_loader, test_loader = get_cifar10_loaders(batch_size=batch)

    use_ln = activation_name == "gelu_ln"
    model = build_mlp(
        depth, hidden=hidden, activation=activation_name, fixed_J=fixed_J, use_ln=use_ln
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)
    loss_fn = nn.CrossEntropyLoss()

    metrics = {
        "train_loss": [],
        "test_acc": [],
        "grad_ratio_epoch5": None,
        "grad_first_epoch5": None,
        "grad_last_epoch5": None,
        "grad_ratio_epoch25": None,
        "grad_first_epoch25": None,
        "grad_last_epoch25": None,
        "J_s_bar_traj": [],
        "K_traj": [],
        "mode_traj": [],
    }

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()

        # Test accuracy
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb).argmax(1)
                correct += (pred == yb).sum().item()
                total += yb.size(0)
        test_acc = correct / total
        metrics["train_loss"].append(round(epoch_loss / len(train_loader), 5))
        metrics["test_acc"].append(round(test_acc, 5))

        # Log CWA stats
        cwa_layers = [m for m in model.modules() if isinstance(m, CWALayer)]
        if cwa_layers:
            J_s_bars = [m._last_J_s_bar for m in cwa_layers]
            Ks = [m._last_K for m in cwa_layers]
            modes = [m._last_mode for m in cwa_layers]
            metrics["J_s_bar_traj"].append(round(float(np.mean(J_s_bars)), 5))
            metrics["K_traj"].append(round(float(np.mean(Ks)), 3))
            metrics["mode_traj"].append(modes[0] if modes else None)

        # Gradient ratio at epoch 5 and final epoch
        if epoch == 5:
            ratio, gf, gl = measure_gradient_ratios(model, train_loader, loss_fn, device)
            metrics["grad_ratio_epoch5"] = round(ratio, 5)
            metrics["grad_first_epoch5"] = round(gf, 8)
            metrics["grad_last_epoch5"] = round(gl, 8)
        if epoch == epochs:
            ratio, gf, gl = measure_gradient_ratios(model, train_loader, loss_fn, device)
            metrics["grad_ratio_epoch25"] = round(ratio, 5)
            metrics["grad_first_epoch25"] = round(gf, 8)
            metrics["grad_last_epoch25"] = round(gl, 8)

        logger.info(f"  Epoch {epoch}/{epochs}: loss={metrics['train_loss'][-1]:.4f}, acc={test_acc:.4f}")

    # Extract final J value for CWA
    cwa_layers = [m for m in model.modules() if isinstance(m, CWALayer)]
    if cwa_layers:
        J_vals = [cwa.get_J().item() for cwa in cwa_layers]
        metrics["final_J_mean"] = round(float(np.mean(J_vals)), 5)
        metrics["fraction_converged_before_Kmax"] = round(
            sum(1 for k in metrics["K_traj"] if k < DEMO_K_MAX) / max(len(metrics["K_traj"]), 1), 4
        )

    metrics["final_test_acc"] = metrics["test_acc"][-1] if metrics["test_acc"] else 0.0
    metrics["final_train_loss"] = metrics["train_loss"][-1] if metrics["train_loss"] else float("nan")
    if metrics["J_s_bar_traj"]:
        metrics["J_s_bar_mean"] = round(float(np.mean(metrics["J_s_bar_traj"])), 5)
    else:
        metrics["J_s_bar_mean"] = None
    if metrics["K_traj"]:
        metrics["K_mean"] = round(float(np.mean(metrics["K_traj"])), 3)
    else:
        metrics["K_mean"] = None

    del model
    if HAS_GPU:
        torch.cuda.empty_cache()
    gc.collect()

    return metrics

## Live Demo Training Run

Runs CWA vs GELU for `DEMO_EPOCHS` epochs at depth `DEMO_DEPTH` with hidden size `DEMO_HIDDEN`. This is a minimal demo — the full experiment ran 25 epochs × 75 configs on GPU.

In [ ]:
demo_results = {}

for act in DEMO_ACTIVATIONS:
    logger.info(f"Training depth={DEMO_DEPTH}, activation={act}, epochs={DEMO_EPOCHS}, hidden={DEMO_HIDDEN}")
    metrics = train_one_config(
        depth=DEMO_DEPTH,
        activation_name=act,
        seed=DEMO_SEED,
        epochs=DEMO_EPOCHS,
        hidden=DEMO_HIDDEN,
        batch=DEMO_BATCH,
        lr=DEMO_LR,
        device=DEVICE,
    )
    demo_results[act] = metrics
    logger.info(f"  Final acc={metrics['final_test_acc']:.4f}, grad_ratio={metrics.get('grad_ratio_epoch25', 'N/A')}")
    if act == 'cwa':
        logger.info(f"  J_s_bar_mean={metrics.get('J_s_bar_mean', 'N/A')}, K_mean={metrics.get('K_mean', 'N/A')}, final_J={metrics.get('final_J_mean', 'N/A')}")

## Precomputed Results: Summary Tables

Load and display the key summary tables from the full 75-run experiment (stored in mini_demo_data.json).

In [ ]:
# Extract summary tables from precomputed data
meta = data['metadata']
tables = meta['summary_tables']
examples = data['datasets'][0]['examples']

# Accuracy by depth
acc_by_depth = tables['accuracy_by_depth']
grad_by_depth = tables['gradient_ratio_by_depth_activation']
fixed_j_grad = tables['fixed_j_gradient_ratios']

print("=" * 60)
print(f"EXPERIMENT: {meta['experiment_name']}")
print(f"VERDICT:    {meta['verdict']}")
print(f"REASON:     {meta['verdict_reason']}")
print("=" * 60)

print("\n--- Accuracy by Depth (mean ± std over 3 seeds) ---")
acts = ['cwa', 'relu', 'gelu', 'selu', 'competing_nl', 'gelu_ln']
print(f"{'Activation':<15}" + "".join(f"  depth {d}" for d in [6, 10, 20]))
for act in acts:
    row = f"{act:<15}"
    for d in [6, 10, 20]:
        d_key = f"depth{d}"
        v = acc_by_depth.get(d_key, {}).get(act, {})
        row += f"  {v.get('mean', 0):.3f}±{v.get('std', 0):.3f}"
    print(row)

print("\n--- Gradient Ratio (epoch 25) by Depth (mean ± std) ---")
print(f"{'Activation':<15}" + "".join(f"  depth {d}" for d in [6, 10, 20]))
for act in acts:
    row = f"{act:<15}"
    for d in [6, 10, 20]:
        d_key = f"depth{d}"
        v = grad_by_depth.get(d_key, {}).get(act, {})
        row += f"  {v.get('mean', 0):.3f}±{v.get('std', 0):.3f}"
    print(row)

print("\n--- Fixed-J Ablation: Gradient Ratio at depth=10 ---")
for j_str, v in fixed_j_grad.items():
    print(f"  {j_str:<12}: {v.get('mean', 0):.4f} ± {v.get('std', 0):.4f}")

print("\n--- Key Findings ---")
for finding in meta.get('key_findings', []):
    print(f"  • {finding}")

## Visualization

Key plots from the full experiment results:
1. Accuracy comparison across depths
2. Gradient ratio comparison across depths
3. Fixed-J ablation: gradient ratio vs J value
4. Demo training curves (CWA vs GELU)

In [ ]:
depths = [6, 10, 20]
activations = ['cwa', 'relu', 'gelu', 'selu', 'competing_nl', 'gelu_ln']
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('CWA vs Baselines: Depth Sweep + Fixed-J Ablation on CIFAR-10', fontsize=13, fontweight='bold')

# ── Plot 1: Accuracy by depth ──────────────────────────────────────────────────
ax = axes[0, 0]
for act, col in zip(activations, colors):
    means = [acc_by_depth.get(f"depth{d}", {}).get(act, {}).get('mean', None) for d in depths]
    stds  = [acc_by_depth.get(f"depth{d}", {}).get(act, {}).get('std',  0.0)  for d in depths]
    valid = [i for i, m in enumerate(means) if m is not None]
    xs = [depths[i] for i in valid]
    ys = [means[i] for i in valid]
    es = [stds[i]  for i in valid]
    lw = 2.5 if act == 'cwa' else 1.2
    ls = '-' if act == 'cwa' else '--'
    ax.errorbar(xs, ys, yerr=es, label=act, color=col, linewidth=lw, linestyle=ls,
                marker='o', markersize=5, capsize=3)
ax.set_xlabel('Depth')
ax.set_ylabel('Test Accuracy')
ax.set_title('Accuracy vs Depth')
ax.set_xticks(depths)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# ── Plot 2: Gradient ratio by depth ───────────────────────────────────────────
ax = axes[0, 1]
for act, col in zip(activations, colors):
    means = [grad_by_depth.get(f"depth{d}", {}).get(act, {}).get('mean', None) for d in depths]
    stds  = [grad_by_depth.get(f"depth{d}", {}).get(act, {}).get('std',  0.0)  for d in depths]
    valid = [i for i, m in enumerate(means) if m is not None]
    xs = [depths[i] for i in valid]
    ys = [means[i] for i in valid]
    es = [stds[i]  for i in valid]
    lw = 2.5 if act == 'cwa' else 1.2
    ls = '-' if act == 'cwa' else '--'
    ax.errorbar(xs, ys, yerr=es, label=act, color=col, linewidth=lw, linestyle=ls,
                marker='o', markersize=5, capsize=3)
ax.axhline(y=2.0, color='red', linestyle=':', alpha=0.7, label='threshold=2.0')
ax.set_xlabel('Depth')
ax.set_ylabel('Gradient Ratio (|log g_first| / |log g_last|)')
ax.set_title('Gradient Ratio vs Depth (lower → more balanced)')
ax.set_xticks(depths)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# ── Plot 3: Fixed-J ablation ───────────────────────────────────────────────────
ax = axes[1, 0]
j_labels = [k for k in fixed_j_grad.keys() if k.startswith('J')]
j_vals_plot = []
j_means = []
j_stds = []
for j_str in sorted(j_labels):
    j_val = float(j_str[1:]) if j_str[1:] != 'learned' else None
    if j_val is not None:
        j_vals_plot.append(j_val)
        j_means.append(fixed_j_grad[j_str]['mean'])
        j_stds.append(fixed_j_grad[j_str]['std'])
ax.errorbar(j_vals_plot, j_means, yerr=j_stds, color='tab:blue', linewidth=2,
            marker='o', markersize=6, capsize=3, label='Fixed-J CWA')
# Mark GELU baseline
gelu_grad_d10 = grad_by_depth.get('depth10', {}).get('gelu', {}).get('mean', None)
if gelu_grad_d10 is not None:
    ax.axhline(y=gelu_grad_d10, color='tab:green', linestyle='--', label=f'GELU (depth10): {gelu_grad_d10:.3f}')
ax.axhline(y=2.0, color='red', linestyle=':', alpha=0.7, label='threshold=2.0')
ax.set_xlabel('Fixed J value')
ax.set_ylabel('Gradient Ratio (epoch 25)')
ax.set_title('Fixed-J Ablation: Gradient Ratio at depth=10')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# ── Plot 4: Demo training curves ───────────────────────────────────────────────
ax = axes[1, 1]
act_colors = {'cwa': 'tab:blue', 'gelu': 'tab:green', 'relu': 'tab:orange'}
for act, metrics in demo_results.items():
    epochs_x = list(range(1, len(metrics['test_acc']) + 1))
    col = act_colors.get(act, 'gray')
    ax.plot(epochs_x, metrics['test_acc'], label=f"{act} (acc={metrics['final_test_acc']:.3f})",
            color=col, linewidth=2, marker='o', markersize=4)
ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy')
ax.set_title(f'Demo Run: depth={DEMO_DEPTH}, hidden={DEMO_HIDDEN}, {DEMO_EPOCHS} epochs')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results_plot.png', dpi=100, bbox_inches='tight')
plt.show()
print("Plot saved to results_plot.png")

## Summary

| Finding | Value |
|---------|-------|
| Verdict | PARTIAL_CONFIRM |
| Mechanism confirmed | J=0.7→grad_ratio=0.364, J=0.9→0.410 (both < 2.0) |
| Accuracy advantage | NOT observed — CWA < GELU at all depths |
| Learned J regime | J·s̄ ≈ 0.285 (sub-critical, IFT branch never triggered) |
| Best at depth 20 | SELU (acc=0.537) due to self-normalizing property |

**Conclusion:** The Curie-Weiss activation's mean-field mechanism does produce more balanced gradient flow (especially at near-critical J), but this does not translate to improved classification accuracy over standard activations like GELU.